In [3]:
import pickle

from neurotools import models
from neurotools import modules
import torch
import torchvision
from torchvision.datasets.mnist import FashionMNIST, MNIST
import networkx as nx
from matplotlib import pyplot as plt

In [4]:
batch_size_train = 1
batch_size_test = 1
train_loader = torch.utils.data.DataLoader(
    FashionMNIST('./tmp/files/', train=True, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_train, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    FashionMNIST('./tmp/files/', train=False, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_test, shuffle=True)

In [5]:
revnet = models.ElegantReverbNetwork(num_nodes=3, input_nodes=(0,), node_shape=(1, 2, 28, 28), edge_module=modules.ElegantReverb, device='cuda')
revnet_decoder = torch.nn.Sequential(torch.nn.MaxPool2d(14),
                                     torch.nn.Conv2d(kernel_size=2, in_channels=1, out_channels=10, device="cuda"))

In [6]:
present_frames = 4
optimizer = torch.optim.Adam(lr=.01, params=list(revnet_decoder.parameters()) + list(revnet.parameters()))
ce_loss = torch.nn.NLLLoss()

In [ ]:
history = []
for epoch in range(2000):
    targets = []
    local_history = []
    optimizer.zero_grad()
    revnet.detach(reset_intrinsic=True)
    for i, (stim, target) in enumerate(train_loader):
        if i > 400:
            break
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[2, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        # insert loss information
        loss_mat = torch.ones(size=decode_input.shape, device='cuda') * (torch.argmax(y_hat) == target)
        revnet.states[2, 0, :, :] = loss_mat
        local_history.append(y_hat.clone())
        targets.append(target.clone())
    lh = torch.cat(local_history[-100:])
    lt = torch.cat(targets[-100:])
    loss = ce_loss(lh, lt)
    acc = torch.count_nonzero(torch.argmax(lh, dim=1) == lt) / 100
    print(loss.detach().cpu().item())
    print(acc.detach().cpu().item())
    loss.backward()
    optimizer.step()
    history.append(acc.detach().cpu().item())

plt.plot(history)
plt.show()

7.822162628173828
0.1599999964237213
7.364126682281494
0.12999999523162842
7.795917987823486
0.14000000059604645
6.613995552062988
0.07000000029802322
5.92385721206665
0.14999999105930328
5.183434009552002
0.11999999731779099
4.648334980010986
0.1599999964237213
4.472647666931152
0.04999999701976776
4.741535186767578
0.04999999701976776
4.4145121574401855
0.07000000029802322
4.104475498199463
0.14000000059604645
4.724301338195801
0.05999999865889549
3.438498020172119
0.08999999612569809
3.627511978149414
0.12999999523162842
3.4959073066711426
0.12999999523162842
3.7050325870513916
0.09999999403953552
3.3967831134796143
0.10999999940395355
2.988409996032715
0.10999999940395355
3.008788824081421
0.07999999821186066
3.169628858566284
0.11999999731779099
2.8173296451568604
0.08999999612569809
2.7809927463531494
0.09999999403953552
2.9202866554260254
0.07999999821186066
2.6830832958221436
0.07999999821186066
2.6464178562164307
0.09999999403953552
2.5822958946228027
0.11999999731779099
2.533

In [6]:
import pickle
with open("./models/l2l_out_mk3.pkl", 'wb') as f:
    pickle.dump((revnet, revnet_decoder), f)

In [6]:
import pickle
with open("./models/l2l_out_mk3.pkl", 'rb') as f:
    revnet, revnet_decoder = pickle.load(f)

In [7]:
# learn without optimization
revnet.detach(reset_intrinsic=True)
local_history = []
targets = []
with torch.no_grad():
    for i, (stim, target) in enumerate(test_loader):
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[0, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        local_history.append(y_hat.clone())
        targets.append(target.clone())

In [9]:
phats = torch.cat(local_history)
t = torch.cat(targets)
f50_acc = torch.count_nonzero(torch.argmax(phats[:10], dim=1) == t[:10]) / 10
l100_acc = torch.count_nonzero(torch.argmax(phats[-100:], dim=1) == t[-100:]) / 100
print("Initial Acc: ", f50_acc.detach().cpu().item())
print("Final Acc: ", l100_acc.detach().cpu().item())

Initial Acc:  0.6000000238418579
Final Acc:  0.5399999618530273


In [11]:
other_loader = torch.utils.data.DataLoader(
    MNIST('./tmp/files/', train=True, download=True,
                               transform=torchvision.transforms.Compose([
                                   torchvision.transforms.ToTensor(),
                                   torchvision.transforms.Normalize(
                                       (0.1307,), (0.3081,))
                               ])),
    batch_size=batch_size_train, shuffle=True)

  0%|          | 0/9912422 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/train-images-idx3-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/28881 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/train-labels-idx1-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/1648877 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/t10k-images-idx3-ubyte.gz to ./tmp/files/MNIST/raw



  0%|          | 0/4542 [00:00<?, ?it/s]

Extracting ./tmp/files/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./tmp/files/MNIST/raw



In [15]:
# learn entirely different set without numerical optimization
revnet.detach(reset_intrinsic=True)
local_history = []
targets = []
with torch.no_grad():
    for i, (stim, target) in enumerate(other_loader):
        if i > 10000:
            break
        for _ in range(present_frames):
            revnet(stim.to("cuda"))
        decode_input = revnet.states[0, 0, :, :][None, None, :, :].clone()
        y_hat = revnet_decoder(decode_input)
        y_hat = y_hat.view(1, 10)
        y_hat = torch.log_softmax(y_hat, dim=1)
        target = target.long().to("cuda")
        local_history.append(y_hat.clone())
        targets.append(target.clone())

In [18]:
phats = torch.cat(local_history)
t = torch.cat(targets)
f50_acc = torch.count_nonzero(torch.argmax(phats[:50], dim=1) == t[:50]) / 50
l100_acc = torch.count_nonzero(torch.argmax(phats[-500:], dim=1) == t[-500:]) / 500
print("Initial Acc: ", f50_acc.detach().cpu().item())
print("Final Acc: ", l100_acc.detach().cpu().item())

Initial Acc:  0.07999999821186066
Final Acc:  0.07800000160932541
